# DEMO sprint 8 - Deel 2 (Grondwaterdroogte indicator)

### Benodigde generieke Python packages

In [ ]:
from pathlib import Path
import pandas as pd
import pastas as ps
import matplotlib.pyplot as plt

## Afleiden van statistiek van de afgelopen 30 jaar
Deze demo maakt gebruik van lokale data (zodat er geen lange reeks met neerslag en verdampingsdata opgevraagd hoeft te worden via de WIWB API)

In [ ]:
data_dir = Path.cwd() / "data_demo"
path_precipitation = data_dir / "reference_precipitation_regions.csv"
path_evaporation = data_dir / "reference_evaporation.csv"
path_info = data_dir / "gwdi_pastas_mapping_all.csv"
path_models = data_dir / "pastas_models"
output_path = data_dir / "df_stats_minima.csv"

Klimaat data inladen

In [ ]:
df_precipitation_regions = pd.read_csv(
    path_precipitation,
    parse_dates=["time"],
    index_col="time",
)
df_evaporation = pd.read_csv(
    path_evaporation,
    parse_dates=["time"],
    index_col="time",
)

Laad specifieke onderdelen CI gwdi

In [ ]:
from toolbox_continu_inzicht.base.adapters.input.pastas_models import (
    input_pastas_models,
)
from toolbox_continu_inzicht.gwdi.processing.gwdi_reference_stats import (
    compute_df_stats_minima,
    prepare_reference_climate,
)

Maak klimaatreeksen aan

In [ ]:
prec100, evap100 = prepare_reference_climate(
    df_precipitation_regions=df_precipitation_regions,
    df_evaporation=df_evaporation,
    precipitation_column="prec_R-R",
    evaporation_column="makkink",
)

Laad locatie-info en Pastas-modellen

In [ ]:
info_locaties = pd.read_csv(path_info)
info_locaties.index = info_locaties["location"].astype(str)
info_locaties["position"] = info_locaties["position"].astype(str)

dict_models_raw = input_pastas_models({"abs_path": path_models})
dict_models_by_location = {
    loc: dict_models_raw[f"{loc}_{info_locaties.loc[loc, 'position']}_tarso"]
    for loc in info_locaties.index
}

Bereken df_stats_minima (grondwater minima statistiek voor elke locatie)

In [ ]:
df_stats_minima = compute_df_stats_minima(
    info_locaties=info_locaties,
    dict_models=dict_models_by_location,
    prec100=prec100,
    evap100=evap100,
    ps_module=ps,
)

# Opslaan
df_stats_minima.to_csv(output_path)

df_stats_minima

## Operationele script gwdi uitvoeren 
De configuratie ziet er als volgt uit:

```yaml
GlobalVariables:
    rootdir: "data_demo"
    calc_time: "2025-06-11 00:00:00"

DataAdapter:
    gwdi_input_precipitation_dynamic:
        type: python
    gwdi_input_evaporation_dynamic:
        type: python

    gwdi_input_climate_sampling_locations:
        type: csv
        file: "gwdi_beemster_loc.csv"

    gwdi_input_pastas_mapping:
        type: csv
        file: "gwdi_pastas_mapping_all.csv"
        dtype:
            location: object
            position: object
    gwdi_input_pastas_models:
        type: pastas_models
        path: "pastas_models"
    gwdi_input_stats_minima:
        type: csv
        file: "df_stats_minima.csv"
        index_col: 0
    gwdi_output:
        type: csv
        file: "gwdi_ouput.csv"
```


### Lees configuratie in en maak dictionary base.Config aan

In [ ]:
from toolbox_continu_inzicht import Config

yaml_config_file = "gwdi_demo.yaml"
path = Path.joinpath(Path.cwd(), "data_demo", yaml_config_file)
config = Config(config_path=path)
config.lees_config()

### Geef config mee aan base.Adapter

In [ ]:
from toolbox_continu_inzicht import DataAdapter

data_adapter_dynamic = DataAdapter(config=config)

Om de pastas modellen correct te runnen moet een langer periode worden mee gegeven.
In dit voorbeeld runnen vanaf 2020 tot 2025.

In [ ]:
path_reference_precipitation_dynamic = (
    data_dir / "reference_da_merged_precipitation.csv"
)
path_reference_evaporation_dynamic = data_dir / "reference_da_merged_evaporation.csv"
path_reference_da_merged_metadata = data_dir / "reference_da_merged_metadata.csv"

reference_da_merged_metadata = pd.read_csv(path_reference_da_merged_metadata)
display(reference_da_merged_metadata)

### Lees de neerslag en verdampingsdata in

In [ ]:
wiwb_out = pd.read_csv(path_reference_precipitation_dynamic, parse_dates=["time"])
knmi_out = pd.read_csv(path_reference_evaporation_dynamic, parse_dates=["time"])

### Combineer de data tot een gezamenlijk (uitgelnd) grid

In [ ]:
wiwb_grid = wiwb_out[["time", "fid"]].drop_duplicates()
knmi_grid = knmi_out[["time", "fid"]].drop_duplicates()
common_grid = wiwb_grid.merge(knmi_grid, on=["time", "fid"], how="inner")

if len(common_grid) == 0:
    raise UserWarning(
        "Geen overlappende (`time`, `fid`)-grid tussen neerslag en verdamping. "
        "Controleer de invoerreeksen."
    )

df_precip_dynamic = wiwb_out.merge(common_grid, on=["time", "fid"], how="inner")
df_evap_dynamic = knmi_out.merge(common_grid, on=["time", "fid"], how="inner")
df_precip_dynamic = df_precip_dynamic.sort_values(["time", "fid"]).reset_index(
    drop=True
)
df_evap_dynamic = df_evap_dynamic.sort_values(["time", "fid"]).reset_index(drop=True)

print("Gezamenlijk grid:")
display(df_precip_dynamic.merge(df_evap_dynamic, on=["time", "fid"]))

### Bereken de grondwaterdroogte-indicator uit

In [ ]:
from toolbox_continu_inzicht.gwdi import (
    GwdiInference,
)

data_adapter_dynamic.set_global_variable("logging", {"level": "CRITICAL"})
data_adapter_dynamic.init_logging(re_initialize=True)

data_adapter_dynamic.set_dataframe_adapter(
    "gwdi_input_precipitation_dynamic",
    df_precip_dynamic,
)
data_adapter_dynamic.set_dataframe_adapter(
    "gwdi_input_evaporation_dynamic",
    df_evap_dynamic,
)

module_dynamic = GwdiInference(data_adapter=data_adapter_dynamic)
module_dynamic.run(
    input=[
        "gwdi_input_precipitation_dynamic",
        "gwdi_input_evaporation_dynamic",
        "gwdi_input_pastas_mapping",
        "gwdi_input_pastas_models",
        "gwdi_input_stats_minima",
    ],
    output="gwdi_output",
)

df_gwdi_dynamic = module_dynamic.df_out.copy()

module_dynamic.data_adapter.set_global_variable("logging", {"level": "CRITICAL"})
module_dynamic.data_adapter.init_logging(re_initialize=True)
module_dynamic._sync_pastas_logger(module_dynamic.data_adapter.logger)
df_gwdi_input_pastas_models = module_dynamic.data_adapter.input(
    "gwdi_input_pastas_models"
)
info_locaties_plot = module_dynamic.data_adapter.input("gwdi_input_pastas_mapping")
peilbuis_index_mapping = dict(
    zip(
        info_locaties_plot["location"],
        range(len(info_locaties_plot)),
    )
)

index_west = peilbuis_index_mapping["Beemster West D_BITA_F-430"]
index_zuid_a = peilbuis_index_mapping["Beemster Zuid A_KR_F-861"]
index_zuid_b = peilbuis_index_mapping["Beemster Zuid B_BITA_F-407"]

### Plot de resultaten van de grondwaterdroogte-indicator in een grafiek

In [ ]:
plt.close("all")
fig, ax = plt.subplots(figsize=(10, 6))
df_gwdi_dynamic["datetime"] = pd.to_datetime(
    df_gwdi_dynamic["datetime"], unit="ms", utc=True
)
df_gwdi_agg = df_gwdi_dynamic.groupby("datetime")["value"].agg(["min", "median", "max"])
ax.fill_between(
    df_gwdi_agg.index,
    df_gwdi_agg["min"],
    df_gwdi_agg["max"],
    alpha=0.2,
    color="grey",
)

df_zuid_a = df_gwdi_dynamic.query("peilbuisid==@index_zuid_a")
ax.plot(
    df_zuid_a["datetime"], df_zuid_a["value"], color="#7d93d3", label="Beemster Zuid A"
)

df_zuid_b = df_gwdi_dynamic.query("peilbuisid==@index_zuid_b")
ax.plot(
    df_zuid_b["datetime"], df_zuid_b["value"], color="#95cd7a", label="Beemster Zuid B"
)

df_west = df_gwdi_dynamic.query("peilbuisid==@index_west")
ax.plot(df_west["datetime"], df_west["value"], color="#fbce69", label="Beemster West D")

ax.plot(df_gwdi_agg.index, df_gwdi_agg["median"], color="k", label="Mediaan")

ax.set_yscale("log")
ax.grid(True, axis="y")
ax.set_ylim([0.1, 150])
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha="right")
ax.legend()

### Plot ook het neerslagtekort

In [ ]:
neeslag_tekort = knmi_out["evaporation"] - wiwb_out["P"]
wiwb_out["p-e"] = neeslag_tekort
wiwb = wiwb_out[wiwb_out["time"] >= "2020-04-01"].copy()
wiwb["neerslag_tekort"] = wiwb.groupby(wiwb["time"].dt.year)["p-e"].cumsum()
wiwb[["time", "fid", "P", "neerslag_tekort"]]
wiwb_2025 = wiwb[wiwb["time"] >= "2025-04-01"]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(wiwb_2025["time"], wiwb_2025["neerslag_tekort"])
ax.set_xlabel("Datum")
ax.set_ylabel("Neerslagtekort (mm)")
ax.set_title("Neerslagtekort vanaf april")
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.grid(True)
plt.tight_layout()
plt.show()